# SENTINEL — train an Overseer on Google Colab T4

**Hackathon submission. End-to-end runnable on a free Colab T4 GPU.**

This notebook trains a 1.7B-parameter "Overseer" model that reviews actions
proposed by an autonomous incident-response agent and decides whether each
action is safe to execute. Training uses **GRPO** (Group Relative Policy
Optimization, [DeepSeekMath, 2024](https://arxiv.org/abs/2402.03300)) against
the [SENTINEL OpenEnv](https://huggingface.co/spaces/Elliot89/sentinel)
running on the Hugging Face Hub.

## What you get when you run this notebook top-to-bottom

| Phase | What happens | Wall clock on T4 |
|---|---|---|
| 0 | Install pinned deps, verify T4 GPU | ~6 min |
| 1 | Wake the SENTINEL OpenEnv on HF Spaces and smoke-test the API | ~1 min |
| 2 | Load Qwen3-1.7B in 4-bit and add a small LoRA adapter | ~3 min |
| 3 | *(optional)* Zero-shot baseline — the F1 we need to beat | ~10 min |
| 4 | SFT warmup — teach the model the JSON output format | ~5 min |
| 5 | GRPO smoke test (5 steps) — proves the training loop works | ~6 min |
| 6 | GRPO training (60 steps by default; bump for stronger results) | ~45 min |
| 7 | Eval the trained adapter, render the comparison plot inline | ~10 min |
| 8 | *(optional)* Push the LoRA adapter to your own HF Hub repo | ~1 min |

> **Free Colab T4 = ~12 h session cap, 16 GB VRAM.** This notebook stays well
> inside both. Every training knob is tunable in **Section 1.5 (Configuration)**
> if you have access to a stronger runtime (L4 / A100 / H100).

## Why a notebook?

The judges asked for a single reviewable artifact that:

1. Documents every step of training a small model with GRPO.
2. Reproduces in one click on commodity hardware (free Colab T4).
3. Surfaces the reward curve and F1 numbers inline so the result is visible
   without leaving the notebook.

Everything heavy (the env, the eval harness, the reward grader, the plotting
helpers) lives in the [GitHub repo](https://github.com/MrEinsteinE/sentinel-openenv)
which this notebook clones in Phase 0. The training loop itself is re-imported
from `training/grpo_hf_job.py` so the Colab path and the HF Jobs production
path share **exactly one** implementation — no drift.

## Before you start

1. **Runtime → Change runtime type → T4 GPU** (free tier is fine).
2. *(Optional)* Add `HF_TOKEN` to **Colab Secrets** (left sidebar → key icon).
   Only needed for the optional Hub-push step at the end.
3. Run cells in order (`Runtime → Run all` also works).


## 1. Bootstrap: clone, install, import

> ⚠️ **Run cells 1.1 → 1.3 in order, top to bottom.** This install pattern
> mirrors the [official Unsloth Qwen3-4B-GRPO Colab notebook](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Qwen3_(4B)-GRPO.ipynb)
> — it is the only configuration verified to work on a free T4 in
> April 2026 without numpy ABI mismatches.

This section has three steps:

1. **1.1** — clone the SENTINEL repo into `/content/sentinel-openenv`.
2. **1.2** — install pinned deps via `uv pip` (Unsloth's recommended
   installer). We pin numpy and pillow to **whatever Colab pre-installed**
   instead of forcing a downgrade — that's the trick that avoids the
   `numpy.dtype size changed (Expected 96, got 88)` error.
3. **1.3** — `import unsloth` once before any project import lands
   `transformers` in `sys.modules`.

The dependency pins are taken straight from Unsloth's published Qwen3 GRPO
notebook so the C extensions and CUDA kernels are guaranteed compatible:

* `vllm==0.9.2` + `triton==3.2.0` — pinned for T4 (sm_75); Unsloth's
  install script branches on `nvidia-smi` and uses these on T4.
* `transformers==4.56.2` — installed in its own line (Unsloth's pattern).
* `trl==0.21.0` — the GRPO callback API used by `grpo_hf_job.py`. Installed
  with `--no-deps` to avoid pulling a transformers upgrade.

### 1.1 Clone the repo


In [ ]:
import os
import pathlib
import shutil
import subprocess
import sys

REPO_URL = os.environ.get("GIT_REPO", "https://github.com/MrEinsteinE/sentinel-openenv")
REPO_DIR = pathlib.Path(os.environ.get("SENTINEL_WORKDIR", "/content/sentinel-openenv"))
BRANCH = os.environ.get("GIT_BRANCH", "main")

if (REPO_DIR / ".git").exists():
    print(f"Repo already at {REPO_DIR}; skipping clone.")
else:
    if REPO_DIR.exists():
        shutil.rmtree(REPO_DIR)
    subprocess.run(
        ["git", "clone", "--depth=1", "--branch", BRANCH, REPO_URL, str(REPO_DIR)],
        check=True,
    )

# Make project imports work and let `grpo_hf_job` skip its own clone step.
os.environ["SENTINEL_WORKDIR"] = str(REPO_DIR)
os.environ["SENTINEL_SKIP_BOOTSTRAP"] = "1"

os.chdir(REPO_DIR)
for p in (str(REPO_DIR), str(REPO_DIR / "training")):
    if p not in sys.path:
        sys.path.insert(0, p)

print("Repo ready at", REPO_DIR)


### 1.2 Install pinned dependencies (official Unsloth pattern)

This is the **exact install pattern** Unsloth uses in their public
[Qwen3-4B-GRPO Colab](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Qwen3_(4B)-GRPO.ipynb).
Critical bits:

* We pin `numpy=={numpy.__version__}` and `pillow=={PIL.__version__}` to
  **whatever Colab pre-installed** instead of forcing a downgrade. Forcing
  numpy<2 used to be the recommended fix, but Unsloth_zoo 2026.4.4 ships
  C extensions built against numpy 2.x — downgrading then triggers
  `numpy.dtype size changed (Expected 96, got 88)`.
* On T4 (compute capability 7.5) we install `vllm==0.9.2` + `triton==3.2.0`.
  vLLM 0.15+ requires sm_80+ and would crash at import.
* `trl==0.21.0` is installed with `--no-deps` so it doesn't drag in a
  newer transformers that breaks Unsloth's allowed window.
* `uv pip` is much faster than `pip` and Unsloth officially supports it.

`%%capture` swallows the noisy install output so you only see errors.


In [ ]:
%%capture
# Install uv (fast pip-compatible resolver) — Unsloth officially supports it.
import os
os.environ["UNSLOTH_VLLM_STANDBY"] = "1"  # 30% extra context length
!pip install --upgrade -qqq uv


In [ ]:
%%capture
# Pin numpy + pillow to whatever Colab pre-installed (NOT downgrade!).
# This is the key insight: trying to downgrade numpy crashes Unsloth_zoo's
# compiled C extensions which were built against numpy 2.x.
import subprocess
try:
    import numpy, PIL
    _numpy = f"numpy=={numpy.__version__}"
    _pil   = f"pillow=={PIL.__version__}"
except Exception:
    _numpy, _pil = "numpy", "pillow"

# T4 (sm_75) needs older vLLM + Triton. Newer GPUs get current versions.
try:
    is_t4 = "Tesla T4" in subprocess.check_output(["nvidia-smi"]).decode()
except Exception:
    is_t4 = False
_vllm, _triton = ("vllm==0.9.2", "triton==3.2.0") if is_t4 else ("vllm==0.15.1", "triton")
print(f"GPU is T4: {is_t4}  ->  installing {_vllm}, {_triton}")

# One install pass, in the order Unsloth's official notebook uses.
!uv pip install --system -qqq --upgrade {_vllm} {_numpy} {_pil} torchvision bitsandbytes xformers unsloth
!uv pip install --system -qqq {_triton}
!uv pip install --system -qqq transformers==4.56.2
!uv pip install --system -qqq --no-deps trl==0.21.0
# SENTINEL-specific extras. peft/accelerate/datasets/huggingface_hub are
# already pulled in by unsloth; we just add the plotting + HTTP-client bits
# that the notebook uses directly.
!uv pip install --system -qqq matplotlib requests pydantic


### 1.3 Import Unsloth FIRST (before any transformers import)

Unsloth patches `transformers` at import time to install its faster CUDA
kernels. If `transformers` lands in `sys.modules` first (which happens the
moment we do `from training.grpo_hf_job import ...`, because that module
needs `transformers.TrainerCallback`), the patches don't apply and you'll
see:

```
WARNING: Unsloth should be imported before [transformers] to ensure all
optimizations are applied.
```

We enforce this from **two** places:

1. The cell below does `import unsloth` once for the kernel.
2. `training/grpo_hf_job.py` itself does `try: import unsloth` before its
   own `transformers` import, so the order is correct even if a future
   cell does `from training.grpo_hf_job import ...` first.

> No kernel restart is needed — Unsloth's official pattern installs
> wheels that are already-compatible with Colab's pre-loaded numpy, so
> there's no ABI mismatch to flush. The `%%capture` blocks above keep
> the noise down.


In [ ]:
import os
import pathlib
import sys

REPO_DIR = pathlib.Path(os.environ.get("SENTINEL_WORKDIR", "/content/sentinel-openenv"))
assert (REPO_DIR / ".git").exists(), (
    f"Repo missing at {REPO_DIR}. Re-run Section 1.1 (git clone)."
)
os.environ["SENTINEL_WORKDIR"] = str(REPO_DIR)
os.environ["SENTINEL_SKIP_BOOTSTRAP"] = "1"
os.chdir(REPO_DIR)
for p in (str(REPO_DIR), str(REPO_DIR / "training")):
    if p not in sys.path:
        sys.path.insert(0, p)

# CRITICAL: import unsloth BEFORE anything that touches transformers.
import unsloth  # noqa: F401

# Now sanity-print what we ended up with so the judges can see the env state.
import numpy as _np
import transformers as _tx
print(f"numpy        : {_np.__version__}")
print(f"transformers : {_tx.__version__}")
print(f"unsloth      : {unsloth.__version__}")
print(f"cwd          : {os.getcwd()}")
print("Bootstrap complete — proceed to Section 1.4.")


### 1.4 Verify the runtime really is a T4 GPU

If this cell prints anything other than a Tesla T4 (or a stronger card), you
can either:

* keep going (the notebook auto-detects compute capability and picks bf16 vs
  fp16 accordingly), or
* `Runtime → Change runtime type → T4 GPU` if you ended up on a CPU runtime.


In [ ]:
import torch

assert torch.cuda.is_available(), "No CUDA GPU detected. Runtime → Change runtime type → T4 GPU."

name = torch.cuda.get_device_name(0)
vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
major, minor = torch.cuda.get_device_capability(0)
print(f"GPU            : {name}")
print(f"VRAM           : {vram_gb:.1f} GB")
print(f"Compute cap.   : {major}.{minor}  ({'bf16-capable' if major >= 8 else 'fp16 only (Turing/T4)'})")
print(f"PyTorch        : {torch.__version__}")


## 1.5 Configuration — every knob you might want to tune

These environment variables are read by `training/grpo_hf_job.py`. Setting
them here keeps both the Colab path and the HF Jobs path on the same code.

### Defaults below are tuned for a free Colab T4 (16 GB VRAM, ~12 h cap)

| Knob | Default here | What it does |
|---|---|---|
| `SENTINEL_USE_VLLM` | `0` | Disable vLLM. Colocated vLLM doesn't fit in 16 GB alongside training. We use plain HuggingFace generate. |
| `SENTINEL_GRPO_NUM_GENERATIONS` | `2` | Group size in GRPO. Smaller = less VRAM per step. (HF Jobs default: 4.) |
| `SENTINEL_GRPO_MAX_COMPLETION_LENGTH` | `256` | Cap on overseer-decision tokens. Most decisions are <100 tokens. (HF Jobs default: 512.) |
| `SENTINEL_GRPO_GRADIENT_ACCUMULATION_STEPS` | `4` | Effective batch = 4 × num_generations. (HF Jobs default: 8.) |
| `SENTINEL_GRPO_MAX_STEPS` | `60` | T4 demo budget. Bump to 200+ on L4/A100. (HF Jobs default: 400.) |
| `SENTINEL_GRPO_SAVE_STEPS` | `10` | How often to checkpoint + render plots. |
| `SENTINEL_GRPO_LOGGING_STEPS` | `1` | Reward/loss logging cadence. |
| `SENTINEL_RUN_ZEROSHOT_EVAL` | `0` | Skip the optional zero-shot baseline by default. Set to `1` in Section 3 to enable. |

If you have an L4 or A100, comment out the T4 line and uncomment the L4/A100
preset — the rest of the notebook will pick up the change.


In [ ]:
import os

# ── T4 (free Colab) defaults ─────────────────────────────────────────────────
os.environ["SENTINEL_USE_VLLM"]                           = "0"
os.environ["SENTINEL_GRPO_NUM_GENERATIONS"]               = "2"
os.environ["SENTINEL_GRPO_MAX_COMPLETION_LENGTH"]         = "256"
os.environ["SENTINEL_GRPO_GRADIENT_ACCUMULATION_STEPS"]   = "4"
os.environ["SENTINEL_GRPO_MAX_STEPS"]                     = "60"
os.environ["SENTINEL_GRPO_SAVE_STEPS"]                    = "10"
os.environ["SENTINEL_GRPO_LOGGING_STEPS"]                 = "1"

# ── L4 / A100 preset (uncomment if you upgraded the runtime) ─────────────────
# os.environ["SENTINEL_USE_VLLM"]                         = "1"
# os.environ["SENTINEL_GRPO_NUM_GENERATIONS"]             = "4"
# os.environ["SENTINEL_GRPO_MAX_COMPLETION_LENGTH"]       = "512"
# os.environ["SENTINEL_GRPO_GRADIENT_ACCUMULATION_STEPS"] = "8"
# os.environ["SENTINEL_GRPO_MAX_STEPS"]                   = "400"

# ── Stable across runtimes ───────────────────────────────────────────────────
os.environ.setdefault("SENTINEL_URL",  "https://elliot89-sentinel.hf.space")
os.environ.setdefault("MODEL_NAME",    "unsloth/Qwen3-1.7B")
os.environ.setdefault("MODEL_REPO",    "Elliot89/sentinel-overseer-qwen3-1.7b")

# Pull HF_TOKEN from Colab Secrets if available (used only by the optional
# Hub-push step at the very end of the notebook).
try:
    from google.colab import userdata  # noqa
    for k in ("HF_TOKEN",):
        try:
            v = userdata.get(k)
            if v:
                os.environ[k] = v
        except Exception:
            pass
except ImportError:
    pass

if os.environ.get("HF_TOKEN"):
    from huggingface_hub import login
    login(token=os.environ["HF_TOKEN"], add_to_git_credential=False)
    print("HF login OK — Hub push at the end will work.")
else:
    print("HF_TOKEN not set — the optional Hub-push step will be skipped.")

print("\nConfig:")
for k in (
    "SENTINEL_URL", "MODEL_NAME", "SENTINEL_USE_VLLM",
    "SENTINEL_GRPO_NUM_GENERATIONS", "SENTINEL_GRPO_MAX_COMPLETION_LENGTH",
    "SENTINEL_GRPO_GRADIENT_ACCUMULATION_STEPS", "SENTINEL_GRPO_MAX_STEPS",
):
    print(f"  {k:50s} = {os.environ.get(k)}")


## 2. Wake the SENTINEL OpenEnv on Hugging Face Spaces

SENTINEL ships as an [OpenEnv](https://github.com/meta-pytorch/OpenEnv) on
Hugging Face Spaces — a FastAPI service that can be driven over HTTP. We
need it for two reasons:

* **Verification** — the cell below polls `/health` until the Space is
  warm (cold start ~60 s) and then hits `/reset` once to confirm the
  request/response shape matches what the trainer expects.
* **Reproducibility** — anyone reading the notebook can open the same Space
  in their browser and play the same scenarios manually in the Gradio
  viewer at the Space's root URL.

> **Implementation note for judges:** the GRPO training loop itself does
> *not* HTTP-call the Space on every step. `make_grpo_dataset` (Section 5)
> walks `server.environment.SentinelEnvironment` **in-process** to
> precompute one (prompt, ground_truth) row per Overseer decision, then
> the reward function grades each rollout in pure Python via
> `graders.grade_overseer_decision`. This is the only way to keep GRPO
> rollout latency reasonable on a free T4 — an HTTP round-trip per
> generation would dominate wall clock.


In [ ]:
from training.grpo_hf_job import warmup_sentinel, build_tool_env_cls, SENTINEL_URL

warmup_sentinel(SENTINEL_URL)

ToolEnv = build_tool_env_cls(SENTINEL_URL)
_env = ToolEnv()
first_obs = _env.reset(task_id="action_screen", seed=1)
print("First observation from /reset (truncated):\n")
print(first_obs[:600])


## 3. Load Qwen3-1.7B in 4-bit and attach a LoRA adapter

We fine-tune **only a LoRA adapter** (rank 16 on the four attention
projections). The base 1.7B Qwen3 stays frozen and 4-bit quantized — that's
what keeps the whole training loop under 16 GB on a T4.

`fast_inference=False` means we do **not** colocate vLLM with the trainer.
On T4 there's not enough VRAM for both the trainer's optimizer state and a
vLLM rollout engine — we use plain `model.generate(...)` instead, and the
GRPO smoke test in Section 5 confirms this still produces a learning signal.


In [ ]:
# ─── SELF-HEALING PRECONDITION CHECK ─────────────────────────────────────────
# This cell historically crashed on Colab T4 with the opaque traceback:
#     ValueError: numpy.dtype size changed, may indicate binary incompatibility.
#                 Expected 96 from C header, got 88 from PyObject
# The error has TWO independent causes and the same surface symptom:
#   (a) transformers was imported before unsloth in a previous cell, so
#       Unsloth's monkey-patches never applied. The "WARNING: Unsloth should
#       be imported before [transformers]" message above is the smoking gun.
#   (b) numpy is at the wrong ABI version. unsloth_zoo 2026.4.4 ships C
#       extensions compiled against numpy 2.x; if anything (an old notebook,
#       a prior install) downgraded numpy to 1.x, unsloth_zoo crashes.
# Once the kernel is in either state, no Python-level code can recover —
# only a kernel restart with the right packages on disk can. This block
# detects the bad state, fixes the on-disk state, and restarts.
import os
import sys
import subprocess

_problems: list[str] = []

if "transformers" in sys.modules:
    _problems.append(
        "transformers is already imported (Unsloth monkey-patches won't apply)"
    )

try:
    import numpy as _np
    if not _np.__version__.startswith("2."):
        _problems.append(
            f"numpy is {_np.__version__} but unsloth_zoo needs >=2.0"
        )
except ImportError:
    _problems.append("numpy is missing entirely")

if _problems:
    print("⚠️  Kernel is in a poisoned state. Self-healing now:")
    for p in _problems:
        print(f"   • {p}")
    # Pin numpy back to 2.x so the post-restart kernel boots cleanly. We
    # don't pin a specific minor version — Unsloth_zoo just needs the 2.x
    # ABI (96-byte dtype struct), and pip will pick whatever current 2.x
    # release is compatible with the rest of the install.
    print("\nReinstalling numpy>=2.0 ...")
    subprocess.run(
        ["pip", "install", "-q", "--upgrade", "--force-reinstall",
         "--no-deps", "numpy>=2.0,<3.0"],
        check=False,
    )
    print("Done. Restarting the Colab kernel — re-run THIS cell after it reconnects.")
    import IPython
    IPython.get_ipython().kernel.do_shutdown(restart=True)
    raise SystemExit("kernel restarting")

# ─── PRECONDITIONS OK — load the model ───────────────────────────────────────
# import unsloth FIRST (it's a no-op if Section 1.3 already ran).
import unsloth  # noqa: F401
from unsloth import FastLanguageModel
import torch

use_vllm = os.environ.get("SENTINEL_USE_VLLM", "0") == "1"

model, tokenizer = FastLanguageModel.from_pretrained(
    os.environ["MODEL_NAME"],
    max_seq_length=2048,    # T4-friendly. HF Jobs default is 4096.
    load_in_4bit=True,
    fast_inference=use_vllm,
)
print(f"Base model loaded. vLLM colocated = {use_vllm}")
print(f"VRAM allocated: {torch.cuda.memory_allocated()/1e9:.2f} GB")


### 3.1 *(optional)* Zero-shot baseline — the F1 we have to beat

This runs the un-trained model against the full 50-scenario held-out split.
On a T4 with HuggingFace generate (no vLLM) it takes ~10 minutes. The
baseline is **not required** for training — it's purely so you have a
"before" number to compare against the trained F1 in Section 7.

We skip it by default. Set `SENTINEL_RUN_ZEROSHOT_EVAL=1` and re-run this
cell if you want the comparison.


In [ ]:
from training.grpo_hf_job import _import_project, run_local_eval

project = _import_project()

baseline_f1 = {}
if os.environ.get("SENTINEL_RUN_ZEROSHOT_EVAL", "0") == "1":
    baseline_summary = run_local_eval(
        model, tokenizer, "qwen3_1_7b_zeroshot", project,
    )
    baseline_f1 = baseline_summary["per_task_f1"]
    print("\nZero-shot per-tier F1:",
          {k: round(v["f1"], 3) for k, v in baseline_f1.items()})
else:
    print("Skipped (SENTINEL_RUN_ZEROSHOT_EVAL != '1').")
    print("The trained-vs-baseline plot in Section 7 will still render the "
          "naive / random / policy-aware reference baselines from eval_data/.")


### 3.2 Attach the LoRA adapter

Rank 16 on `q_proj`, `k_proj`, `v_proj`, `o_proj`. Trainable parameter
count is ~2 M — 0.1% of the base model. Unsloth's gradient checkpointing
cuts activation memory by another ~30%.


In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    use_gradient_checkpointing="unsloth",
    random_state=42,
)
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f"LoRA attached. Trainable: {trainable:,} / {total:,} "
      f"({100 * trainable / total:.3f}%)")


## 4. SFT warmup — teach the model the JSON output format

GRPO can only reinforce behaviour the model is already capable of producing.
Out-of-the-box, Qwen3-1.7B doesn't emit the strict JSON
`{"decision": "...", "justification": "..."}` shape that the SENTINEL grader
expects, so we run **one epoch** of supervised fine-tuning on
`training/sft_data/sft_warmup.jsonl` first.

That file holds 2,623 (prompt, completion) pairs mined from the
policy-aware heuristic — the same baseline we're trying to beat. SFT here
is purely about format compliance; the GRPO step that follows is what
actually teaches the model **which** decision to pick.

On a T4 this takes ~5 min for 1 epoch.


In [ ]:
from training.grpo_hf_job import run_sft

run_sft(model, tokenizer, epochs=1, output_dir="outputs/sft_warmup_1ep")
print("SFT warmup complete.")


## 5. GRPO smoke test (5 steps) — gate the long run

Before we burn an hour of T4 time on the long run, we run **5 GRPO steps**
to confirm:

1. Reward variance is non-zero (the binary grader fires at least once).
2. The policy isn't saturated (at least one log entry below 1.0 — otherwise
   GRPO has no signal to optimize against).
3. Per-step wall clock is under 90 s (otherwise the long run won't fit).

If any of those three fail, the assert at the end of this cell will fire
and the next sections won't run. The most common failure on T4 is "no
reward signal" — fix is usually to re-run Section 4 with `epochs=2`.


In [ ]:
from pathlib import Path
from training.grpo_hf_job import (
    TrackingCallback, _build_grpo_trainer, make_grpo_dataset,
    GRPO_CONFIG, PLOTS_DIR, CKPT_DIR, SMOKE_STEPS,
)

PLOTS_DIR.mkdir(parents=True, exist_ok=True)
CKPT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Building smoke dataset (n=64 proposals from action_screen seeds)...")
smoke_ds = make_grpo_dataset(n_samples=64)

smoke_cb = TrackingCallback(
    plots_dir=PLOTS_DIR,
    ckpt_dir=CKPT_DIR,
    model=model,
    plot_loss_fn=project["plot_loss"],
    plot_reward_fn=project["plot_reward"],
    plot_every=GRPO_CONFIG["save_steps"],
    is_smoke=True,
)
smoke_trainer = _build_grpo_trainer(
    model, tokenizer, smoke_ds, smoke_cb,
    output_dir="outputs/grpo_smoke",
    max_steps=SMOKE_STEPS,
    use_vllm=use_vllm,
)
smoke_trainer.train()

ok, msg = smoke_cb.smoke_pass()
print("\nSmoke result:", msg)
assert ok, "Smoke test failed. Re-run Section 4 with epochs=2 and try again."
print("Smoke OK — proceeding to the long run.")


## 6. GRPO training (60 steps default; bump for stronger results)

This is the actual training run. With the T4 defaults from Section 1.5
(`max_steps=60`, `num_generations=2`, `max_completion_length=256`,
`gradient_accumulation_steps=4`) it takes roughly **45 minutes** end to end.

Plots and checkpoints land in `training/plots/` and `training/checkpoints/`
every `SENTINEL_GRPO_SAVE_STEPS` steps. The notebook displays them inline at
the end of Section 7.

### Auto-abort safety net

`TrackingCallback` watches the rolling-mean reward and aborts early if
training stalls:

* **Step 100, mean reward < 0.05** → fall back to `step100_resft` (re-train
  SFT). On T4 we don't reach step 100 by default, so this never fires.
* **Step 200, mean reward < 0.85** → fall back to `step200_sft_only` (use
  the SFT checkpoint as the published model).

These thresholds are read from `STEP100_MIN_REWARD` / `STEP200_MIN_REWARD`
at import time. The published `Elliot89/sentinel-overseer-qwen3-1.7b`
adapter (F1=0.980 on the 50-scenario held-out split) was actually trained
this way — GRPO ran 200 steps, didn't beat the SFT baseline by the required
margin, so the SFT checkpoint was published. Treat the published headline as
an SFT result until a future GRPO run survives the abort.


In [ ]:
n_long = GRPO_CONFIG["max_steps"]
print(f"Building long dataset (n={n_long * GRPO_CONFIG['gradient_accumulation_steps']} proposals)...")
long_ds = make_grpo_dataset(
    n_samples=n_long * GRPO_CONFIG["gradient_accumulation_steps"],
)

long_cb = TrackingCallback(
    plots_dir=PLOTS_DIR,
    ckpt_dir=CKPT_DIR,
    model=model,
    plot_loss_fn=project["plot_loss"],
    plot_reward_fn=project["plot_reward"],
    plot_every=GRPO_CONFIG["save_steps"],
)
long_trainer = _build_grpo_trainer(
    model, tokenizer, long_ds, long_cb,
    output_dir="outputs/grpo_long",
    max_steps=n_long,
    use_vllm=use_vllm,
)
long_trainer.train()

print(f"\nFinished {n_long} GRPO steps.")
print(f"Best reward window: {long_cb.best_reward:.3f} at step {long_cb.best_step}")
print(f"Abort path        : {long_cb.abort_reason or '(none — full run completed)'}")


## 7. Evaluate the trained adapter + render the comparison plot

We:

1. Save the trained LoRA adapter under `training/checkpoints/qwen3-1.7b-sentinel-best/`.
2. Run the full held-out eval (50 scenarios across `action_screen`,
   `war_room`, `drift_ops`) using the same `run_local_eval` harness as the
   HF Jobs entrypoint.
3. Stack the result onto the per-baseline F1 numbers already in
   `eval_data/baseline_*.json` (naive, random, policy-aware, the published
   Qwen3 model, etc.) and render `baseline_vs_trained.png`.
4. Display all three plots inline so the judges don't need to dig.


In [ ]:
from training.grpo_hf_job import EVAL_DIR, _load_baselines

final_dir = CKPT_DIR / "qwen3-1.7b-sentinel-best"
final_dir.mkdir(parents=True, exist_ok=True)
model.save_pretrained(str(final_dir))
tokenizer.save_pretrained(str(final_dir))
print(f"Saved adapter -> {final_dir}")

print("\nRunning trained-model eval on 50 held-out scenarios...")
trained_summary = run_local_eval(
    model, tokenizer, "trained_qwen3_1_7b_grpo", project,
)
f1_per_tier = trained_summary["per_task_f1"]

baselines = _load_baselines(EVAL_DIR)
if baseline_f1:
    baselines["qwen3_1_7b_zeroshot"] = baseline_f1
baselines["trained_qwen3_1_7b_grpo"] = f1_per_tier

project["plot_baseline_vs_trained"](
    baselines,
    trained_label="trained_qwen3_1_7b_grpo",
    out_path=str(PLOTS_DIR / "baseline_vs_trained.png"),
    tier="action_screen",
)


In [ ]:
from IPython.display import Image, display, Markdown

display(Markdown("### Reward + loss curves"))
for name in ("grpo_loss.png", "grpo_reward.png"):
    p = PLOTS_DIR / name
    if p.exists():
        display(Image(filename=str(p)))

display(Markdown("### Trained vs. baseline F1 (action_screen tier)"))
p = PLOTS_DIR / "baseline_vs_trained.png"
if p.exists():
    display(Image(filename=str(p)))

display(Markdown("### Headline numbers"))
print(f"Zero-shot action_screen F1 : "
      f"{baseline_f1.get('action_screen', {}).get('f1', float('nan')):.3f}"
      f"  (skipped if Section 3.1 was off)")
print(f"Trained   action_screen F1 : {f1_per_tier['action_screen']['f1']:.3f}")
print(f"Trained   war_room     F1  : {f1_per_tier['war_room']['f1']:.3f}")
print(f"Trained   drift_ops    F1  : {f1_per_tier['drift_ops']['f1']:.3f}")


## 8. *(Optional)* Push the LoRA adapter to your own HF Hub repo

Skip this section unless you've set `MODEL_REPO` to a repo you own and
added an `HF_TOKEN` with `write` scope to Colab Secrets. The default
`MODEL_REPO` (`Elliot89/sentinel-overseer-qwen3-1.7b`) is the published
hackathon checkpoint and you won't have permission to overwrite it.

To push to your own:

```python
os.environ["MODEL_REPO"] = "<your-username>/sentinel-overseer-qwen3-1.7b"
```

…then run the cell.


In [ ]:
from training.grpo_hf_job import _write_summary, push_lora_to_hub
import time

_write_summary(
    f1_per_tier=f1_per_tier,
    baseline_f1=baseline_f1,
    abort_path=long_cb.abort_reason,
    wall_clock_s=time.time(),
    best_step=long_cb.best_step,
)

if os.environ.get("HF_TOKEN"):
    url = push_lora_to_hub(final_dir)
    print(f"Adapter pushed -> {url}")
else:
    print("HF_TOKEN not set — Hub push skipped. Adapter still on disk at:")
    print(f"  {final_dir}")


---

### That's it

If you only care about the headline number, scroll up to the
`Trained vs. baseline F1` plot in Section 7. The full per-scenario
JSON for the trained run is in `eval_data/baseline_trained_qwen3_1_7b_grpo.json`,
and `training/run_summary.json` has the configuration that produced it.

Questions / repro issues:
[github.com/MrEinsteinE/sentinel-openenv](https://github.com/MrEinsteinE/sentinel-openenv)
